# Short-Term Memory

In [1]:
import dotenv
from agents import Agent, Runner, SQLiteSession, trace

dotenv.load_dotenv()

True

In [2]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant comparing how healthy different foods are.
    If you answer, give a list of how healthy the foods are with a score from 1 to 10. Order by: healtiest food comes first.

    Example:
    Q: Compare X and Y
    A: X is healtier as Y.
    1) X: 8/10 - Very healthy but high in fructose
    2) Y: 3/10 - High in sugar and fat
    """,
)

## No Memory

In [3]:
result = await Runner.run(nutrition_agent, "Which is healthier, bananas or lollipop?")
print(result.final_output)

1) Bananas: 8/10 - Good fiber, potassium, vitamins; natural sugars but overall nutritious.

2) Lollipop: 2/10 - Mostly empty calories from sugar; little to no nutrients.


now we try to add another food to the comparison. But without memory, it cannot add apple to the comparison as we can see running the next block.

In [4]:
result = await Runner.run(nutrition_agent, "Add apples to the comparison")
print(result.final_output)

Which other foods should apples be compared to? Please provide the list of foods to include, and I’ll rank them with scores.


## Short Term Memory
built in database is sqlite database. it can be triggered as follows:

In [5]:
session = SQLiteSession("conversation_history")

In [6]:
result = await Runner.run(
    nutrition_agent, "Which is healthier, bananas or lollipop?"
)
print(result.final_output)

Bananas are healthier than lollipops.

1) Bananas: 8/10 - Good fiber, potassium, vitamins; natural sugars but nutrient-dense.
2) Lollipop: 2/10 - Mostly sugar; minimal nutrients.


just adding the sqlite database is of no use, it still starts a new session.

In [7]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(
        nutrition_agent, "Add apples to the comparison"
    )

print(result.final_output)

Please specify the other foods to compare apples against (e.g., bananas, oranges, cookies, almonds). Once you provide them, I’ll rank by healthiness 1–10.


we also need to add the session that the agent is supposed to use i.e. refer to in our example.

In [8]:
result = await Runner.run(
    nutrition_agent, "Which is healthier, bananas or lollipop?", session=session
)
print(result.final_output)

1) Bananas: 8/10 - Nutritious: potassium, fiber, vitamin C; natural sugars but overall balanced.
2) Lollipop: 2/10 - Mostly sugar with little to no nutrients; high added sugar.


In [9]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(
        nutrition_agent, "Add apples to the comparison", session=session
    )

print(result.final_output)

1) Bananas: 8/10 - Nutritious: potassium, fiber, vitamin C; natural sugars but overall balanced.
2) Apples: 7/10 - Good fiber, vitamin C, and polyphenols; lower calories, natural sugars.
3) Lollipop: 2/10 - Mostly sugar with little to no nutrients; high added sugar.


As we can see through the result, adding session to the query for reference helped us to add another object to comparison without losing the context of the prior comparison.